In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

## 1- Business Overview &Target 

Objective: To develop and evaluate a Convolutional Neural Network (CNN)-based image classification model that identifies 4 classes from a labelled dataset, achieving high accuracy and F1 score with strong generalisation across varied capture conditions; and ultimately to deploy the model for real-time species identification in web or mobile applications.

Target :Target :Target :Model Building: Construct a custom CNN from scratch with at least 5 Convolutional layers and 3 Pooling layers.
Optimization: Apply Regularization (like Dropout) to ensure the model performs well on new, unseen data.
Deployment: Create an interactive Streamlit web application and host it on HuggingFace for real-time image prediction.
Evaluation: Compare the "from-scratch" model's performance against a Transfer Learning (e.g., VGG16) approach

Note :The grape leaves dataset that we used has 12.000 total images, divided into four classes, with 3.000 images per class: Healthy, Black Rot, ESCA, and Leaf Blight.

## 2- Importing Libraries

In [ ]:
import numpy as np           
import pandas as pd          
import os                    
import cv2
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import Sequential       
from tensorflow.keras.utils import to_categorical   
from tensorflow.keras.layers import Dense, Conv2D, InputLayer, Reshape, MaxPooling2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
import warnings
warnings.filterwarnings('ignore')

## 3- Reading Dataset

In [ ]:
img_path='/kaggle/input/augmented-grape-disease-detection-dataset/Final Training Data/'

In [ ]:
os.listdir(img_path)

In [ ]:
labels=['ESCA','Healthy','Leaf Blight','Black Rot']

In [ ]:
img_list=[]
label_list=[]
for label in labels:
    for img_file in os.listdir(img_path+label):
        img_list.append(img_path+label+"/"+img_file)
        label_list.append(label)

In [ ]:
df=pd.DataFrame({'img':img_list,'label':label_list})

In [ ]:
df.sample(10)

In [ ]:
d={'ESCA':0,'Healthy':1,'Leaf Blight':2,'Black Rot':3}

In [ ]:
df['label_encoded']=df['label'].map(d)

In [ ]:
df.sample()

In [ ]:
plt.figure(figsize=(10, 8))
df['label'].value_counts().plot.pie(autopct='%1.1f%%', shadow=True, cmap='Spectral', startangle=140)
plt.title("Grape Disease Detection %")
plt.ylabel('')
plt.show()

In [ ]:
images = []
for label in labels:
    img_path = df[df['label'] == label]['img'].iloc[0]
    image = Image.open(img_path)
    images.append((image, label))
plt.figure(figsize=(6,6))
for i, (image, label) in enumerate(images):
    plt.subplot(2, 2, i + 1)
    plt.imshow(image)
    plt.title(label)    
    plt.axis('off')
plt.tight_layout()
plt.show()

## 4-DATA Processing

In [ ]:
x=[]
for img in df['img']:
    img=cv2.imread(str(img))
    img=cv2.resize(img,(170,170))
    img=img/255.0
    x.append(img)

In [ ]:
x = np.array(x)

In [ ]:
x.shape

In [ ]:
y=df[['label_encoded']]

## 5- CNN Modelling

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.2,random_state=42)

In [ ]:
from tensorflow.keras.layers import Input
model = Sequential()

model.add(Input(shape=(170,170,3)))

model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))


model.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(Conv2D(128, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

model.add(Conv2D(256, (3,3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

model.add(Flatten())

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(25, activation='softmax'))

model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
early_stop = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [ ]:
history = model.fit(x_train, y_train,validation_data=(x_test, y_test),epochs=25,batch_size=32,callbacks=[early_stop])

In [ ]:
model.save("grape_disease.keras")

## 6-Model Interpretation

In [ ]:
history.history['accuracy'][-1]

In [ ]:
plt.plot(history.history['accuracy'],label='Accuracy')
plt.plot(history.history['val_accuracy'],label='Val_Accuracy')
plt.legend();

In [ ]:
plt.plot(history.history['loss'], label='Loss')
plt.plot(history.history['val_loss'], label='Val_Loss')
plt.legend();

## 7- Transfer Learning

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.optimizers import Adam

In [ ]:
data_dir = '/kaggle/input/augmented-grape-disease-detection-dataset/Final Training Data/'
img_size = 224

datagen = ImageDataGenerator(preprocessing_function=preprocess_input,validation_split=0.20)

train_gen = datagen.flow_from_directory(data_dir,target_size=(img_size, img_size),class_mode='sparse',subset='training',shuffle=True)
val_gen = datagen.flow_from_directory(data_dir,target_size=(img_size, img_size),class_mode='sparse',subset='validation',shuffle=False)

In [ ]:
base_model = MobileNetV2(weights='imagenet',include_top=False,input_shape=(img_size, img_size, 3))
base_model.trainable = False

In [ ]:
from tensorflow.keras.layers import (Dense,Dropout,GlobalAveragePooling2D)
model=Sequential()
model.add(base_model)  
model.add(GlobalAveragePooling2D())
model.add(Dense(512,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(25,activation='softmax'))

model.compile(optimizer=Adam(learning_rate=0.0001),loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
early_stop = EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [ ]:
history2=model.fit(train_gen,epochs=50,validation_data=val_gen, callbacks=[early_stop])

In [ ]:
history2.history['accuracy'][-1]

## 8- Model Interpretation for TL 

In [ ]:
# Accurancy
plt.figure(figsize=(8,5))
plt.plot(history2.history['accuracy'], label='Train Accuracy')
plt.plot(history2.history['val_accuracy'], label='Validation Accuracy')

plt.title("MobileNetV2 Transfer Learning Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Loss
plt.figure(figsize=(8,5))
plt.plot(history2.history['loss'], label='Train Loss')
plt.plot(history2.history['val_loss'], label='Validation Loss')

plt.title("MobileNetV2 Transfer Learning Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## 9- Confusion Matrix for TL

In [ ]:
tahmin = model.predict(val_gen)
tahmin = np.argmax(tahmin, axis=1)

y_test = val_gen.classes

labels = list(val_gen.class_indices.keys())

plt.figure(figsize=(12,8))
sns.heatmap(confusion_matrix(y_test, tahmin),annot=True,cmap='Blues',xticklabels=labels,yticklabels=labels)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Transfer Learning Confusion Matrix")
plt.show()

In [ ]:
model.summary()

## 10- Conclusion& Evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

val_gen.reset()

x_all = []
y_all = []

for i in range(len(val_gen)):
    x_batch, y_batch = val_gen[i]
    x_all.append(x_batch)
    y_all.append(y_batch)

x_all = np.vstack(x_all)
y_all = np.hstack(y_all)

class_names = list(val_gen.class_indices.keys())
num_classes = len(class_names)

plt.figure(figsize=(20,20))

for class_id in range(num_classes):
    
    # O sınıfa ait ilk indexi bul
    idx = np.where(y_all == class_id)[0][0]
    
    img_display = (x_all[idx] + 1) / 2
    img_display = np.clip(img_display, 0, 1)
    
    prediction = model.predict(np.expand_dims(x_all[idx], axis=0))
    predicted_label = np.argmax(prediction)
    
    plt.subplot(5,5,class_id+1)
    plt.imshow(img_display)
    plt.title(
        f"Actual: {class_names[class_id]}\nPredicted: {class_names[predicted_label]}",
        fontsize=10
    )
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
model.save('grape_disease_mobilenet_98.keras')

In [ ]:
import json
labels_dict = {v: k for k, v in train_gen.class_indices.items()}
with open('labels.json', 'w') as f:
    json.dump(labels_dict, f)

In this project, I performed a comparative analysis of two distinct Deep Learning approaches to detection grape disease:

Custom CNN Architecture: I initially designed a baseline Convolutional Neural Network (CNN) with an input resolution of 170*170 pixels. This model achieved an accuracy of only 87%.

Transfer Learning (MobileNetV2): In the second phase, I increased the image size to 224x224 pixels and utilized the pre-trained MobileNetV2 architecture. By leveraging the power of Transfer Learning and specific preprocess_input optimizations, the model reached a remarkable 99% accuracy rate, successfully completing the project.
